# META-CXR Table 5 — Mean F1 từ GCS checkpoints

Notebook reproduce **Table 5** từ paper *Chest X-Ray Report Generation Using Abnormality Guided Vision Language Model* (IEEE Access 2025, DOI 10.1109/ACCESS.2025.3606961).

**Bảng**: `Mean F1 score across 5 common abnormalities` (Atelectasis, Cardiomegaly, Consolidation, Edema, Pleural Effusion) cho mỗi combo của 3 vision encoder **RN50 (BioViL-T) / ViT (PubMedCLIP) / Swin**.

**Checkpoints**: pull từ `gs://meta-cxr-checkpoint/<run>/checkpoint_best.pth` — 7 run tương ứng 7 combo (2³−1).

## Yêu cầu trên Kaggle

1. **Kaggle Secrets** → thêm key `GCS_SERVICE_ACCOUNT` chứa service-account JSON. Cũng hỗ trợ `GCP_SERVICE_ACCOUNT_JSON` hoặc `GCP_SERVICE_ACCOUNT_B64`. Service account cần quyền `roles/storage.objectViewer` trên bucket `meta-cxr-checkpoint`.
2. Attach Kaggle datasets:
   - `/kaggle/input/datasets/phuong20052/mimic-cxr-jpg-lite` — ảnh + CheXpert CSV + metadata CSV
   - `/kaggle/input/datasets/phuong20052/mimic-cxr-p10-processed` — `train.csv`, `val.csv`, `test.csv`
   - Dataset chứa source code `META-CXR` (hoặc notebook chạy trong repo đã clone vào working).
3. Bật **Internet** trong Settings để Google Cloud Storage client và `pip install` hoạt động. Cell dependency pin `numpy>=2,<2.3` và `pandas>=2.2,<3`.
4. GPU **T4 ×2** hoặc **P100**.

## Output

Bảng 7 hàng với các cột: `RN50`, `ViT`, `Swin`, `Mean F1 Score`, `Paper F1`, `Δ vs Paper`. Hàng nào không có `checkpoint_best.pth` trên GCS → cột `Mean F1 Score` ghi `MISSING` (in-place skip).

## Lưu ý F1 metric

Notebook dùng `f1_score(average='weighted', zero_division=1)` (giống `META_CXR_encoder_f1_table_kaggle.ipynb`). Nếu bạn muốn paper-faithful binary P-vs-rest macro F1 (giống `eval_paper_style.py`) thì đổi block `per_task[...] = f1_score(...)` trong cell định nghĩa `mean_f1_for_run`.

In [7]:
"""
Cell 1 — Install dependencies (Kaggle).

Chạy cell này đầu tiên sau khi Restart Kernel. Cell này giữ numpy và
pandas ở major version 2 để tránh lệch ABI với Kaggle/Pandas mới.
"""
import shutil
import subprocess
import sys
from pathlib import Path

# if any(name in sys.modules for name in ("numpy", "pandas", "torch", "cv2")):
#     raise RuntimeError(
#         "Restart Kernel rồi chạy lại Cell 1 trước mọi cell import numpy/pandas/torch/cv2."
#     )


def pip_install(*packages):
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q", "--upgrade",
            "--upgrade-strategy", "only-if-needed", *packages,
        ],
        check=True,
    )


PACKAGES = [
    # OpenCV 4.12 requires numpy>=2,<2.3 on Python >=3.9, so keep NumPy 2 but below 2.3.
    "opencv-python>=4.12,<4.13",
    "scikit-image>=0.22",
    "scikit-learn>=1.4",
    "omegaconf==2.3.0",
    "pycocoevalcap",
    "torchinfo",
    "wandb",
    "loralib==0.1.1",
    "iterative-stratification",
    "iopath",
    "hi-ml-multimodal==0.2.2",
    "timm>=0.9.0",
    "spacy>=3.8,<3.9",
    "nltk>=3.9",
    "google-cloud-storage",
    "transformers==4.44.2",
]
pip_install(*PACKAGES)
pip_install("git+https://github.com/huggingface/peft.git@e536616888d51b453ed354a6f1e243fecb02ea08")
pip_install(
    "https://github.com/explosion/spacy-models/releases/download/"
    "en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl"
)

import nltk
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

java_bin = shutil.which("java")
JAVA_HOME_DETECTED = (
    str(Path(java_bin).resolve().parent.parent)
    if java_bin
    else "/usr/lib/jvm/java-11-openjdk-amd64"
)
print(f"Detected JAVA_HOME: {JAVA_HOME_DETECTED}")

import cv2 as _cv2
import numpy as _np
import pandas as _pd
import torch as _torch

if not _np.__version__.startswith("2."):
    raise RuntimeError(f"numpy phải là major version 2, hiện tại là {_np.__version__}")
if not _pd.__version__.startswith("2."):
    raise RuntimeError(f"pandas phải là major version 2, hiện tại là {_pd.__version__}")

print(f"numpy   = {_np.__version__}")
print(f"pandas  = {_pd.__version__}")
print(f"opencv  = {_cv2.__version__}")
print(f"torch   = {_torch.__version__}")
print(f"GPUs available: {_torch.cuda.device_count()}")
for _i in range(_torch.cuda.device_count()):
    print(f"  GPU {_i}: {_torch.cuda.get_device_name(_i)}")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 77.8 MB/s eta 0:00:00
Detected JAVA_HOME: /usr/lib/jvm/java-17-openjdk-amd64
numpy   = 2.0.2
pandas  = 2.3.3
opencv  = 4.12.0
torch   = 2.10.0+cu128
GPUs available: 2
  GPU 0: Tesla T4
  GPU 1: Tesla T4


In [8]:
import os

REPO_DIR = "/kaggle/working/META-CXR"

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/minhphuong150505/Meta-CXR-Kaggle.git {REPO_DIR}
else:
    print(f"Repository already exists at {REPO_DIR}, pulling latest changes...")
    !git -C {REPO_DIR} pull

# Change working directory to repo root
os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")
!ls -la

Repository already exists at /kaggle/working/META-CXR, pulling latest changes...
Already up to date.
Working directory: /kaggle/working/META-CXR
total 404
drwxr-xr-x 13 root root  4096 May 25 09:19 .
drwxr-xr-x  4 root root  4096 May 25 09:19 ..
drwxr-xr-x  2 root root  4096 May 25 09:19 assets
drwxr-xr-x  3 root root  4096 May 25 09:19 biovil_t
-rw-r--r--  1 root root   538 May 25 09:19 build_container.sh
drwxr-xr-x  3 root root  4096 May 25 09:19 checkpoints
-rw-r--r--  1 root root  8919 May 25 09:19 CHECKPOINT_WORKFLOW.md
drwxr-xr-x  4 root root  4096 May 25 09:19 cloud
drwxr-xr-x  2 root root  4096 May 25 09:19 configs
-rw-r--r--  1 root root    90 May 25 09:19 Dockerfile
-rw-r--r--  1 root root  2612 May 25 09:19 eval_guild
-rw-r--r--  1 root root 10120 May 25 09:19 eval_paper_style.py
-rw-r--r--  1 root root  7474 May 25 09:19 generate_mimic_cxr_cleaned.ipynb
drwxr-xr-x  8 root root  4096 May 25 09:21 .git
-rw-r--r--  1 root root   247 May 25 09:19 .gitignore
-rw-r--r--  1 root r

In [9]:
import os
import sys
import shutil
from pathlib import Path

WORK_DIR = Path('/kaggle/working')
INPUT_DIR = Path('/kaggle/input')
KAGGLE_DATASETS_BASE = Path('/kaggle/input/datasets/phuong20052')

def is_meta_cxr_project(path: Path) -> bool:
    return (path / 'model' / 'lavis').exists() and (path / 'pretraining').exists()

project_candidates = [Path.cwd(), WORK_DIR / 'META-CXR', KAGGLE_DATASETS_BASE, KAGGLE_DATASETS_BASE / 'META-CXR']
if INPUT_DIR.exists():
    for root in INPUT_DIR.glob('*'):
        project_candidates.extend([root, root / 'META-CXR'])
if KAGGLE_DATASETS_BASE.exists():
    for root in KAGGLE_DATASETS_BASE.glob('*'):
        project_candidates.extend([root, root / 'META-CXR'])

source_project = next((p for p in project_candidates if is_meta_cxr_project(p)), None)
if source_project is None:
    raise FileNotFoundError('Không tìm thấy code META-CXR. Hãy attach dataset/source chứa thư mục META-CXR.')

PROJECT_DIR = WORK_DIR / 'META-CXR'
if source_project.resolve() != PROJECT_DIR.resolve():
    if PROJECT_DIR.exists() and is_meta_cxr_project(PROJECT_DIR):
        pass
    else:
        ignore = shutil.ignore_patterns('.git', 'wandb', '__pycache__', '*.pyc', 'output', 'outputs', 'checkpoints')
        shutil.copytree(source_project, PROJECT_DIR, dirs_exist_ok=True, ignore=ignore)

os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))
sys.path.insert(0, str(PROJECT_DIR / 'model'))

def first_existing(paths):
    for item in paths:
        path = Path(item)
        if path.exists():
            return path
    raise FileNotFoundError('Không tìm thấy path nào trong: ' + ', '.join(map(str, paths)))

IMAGE_ROOT = first_existing([
    KAGGLE_DATASETS_BASE / 'mimic-cxr-jpg-lite',
    '/kaggle/input/mimic-cxr-jpg-lite',
    '/kaggle/input/datasets/mimic-cxr-jpg-lite',
])
PROCESSED_ROOT = first_existing([
    KAGGLE_DATASETS_BASE / 'mimic-cxr-p10-processed',
    '/kaggle/input/mimic-cxr-p10-processed',
    '/kaggle/input/datasets/mimic-cxr-p10-processed',
])

# Checkpoints are pulled from GCS into Kaggle temp storage, matching the training notebook.
CHECKPOINT_ROOT = Path('/kaggle/temp/checkpoints')
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)

java_home = globals().get('JAVA_HOME_DETECTED')
if not java_home:
    import subprocess
    result = subprocess.run(
        "readlink -f $(which java) | sed 's|/bin/java||'",
        shell=True,
        capture_output=True,
        text=True,
    )
    java_home = result.stdout.strip() or '/usr/lib/jvm/java-11-openjdk-amd64'
java_path = java_home + '/bin:'

GCS_PROJECT = os.environ.get('GCS_PROJECT', 'mimic-cxr-jpg-491409')
GCS_BUCKET = os.environ.get('GCS_BUCKET', 'gs://meta-cxr-checkpoint')

(PROJECT_DIR / 'configs').mkdir(exist_ok=True)
(PROJECT_DIR / 'configs' / 'env_config.yaml').write_text(f'''paths:
  data_root: "{IMAGE_ROOT}"
  mimic_cxr_jpg_root: "{IMAGE_ROOT}"
  split_csv: "{IMAGE_ROOT}/mimic-cxr-2.0.0-split.csv"
  reports_csv: "/kaggle/working/mimic_cxr_cleaned.csv"
  chexpert_csv: "{IMAGE_ROOT}/mimic-cxr-2.0.0-chexpert.csv"
  metadata_csv: "{IMAGE_ROOT}/mimic-cxr-2.0.0-metadata.csv"
  processed_dir: "{PROCESSED_ROOT}"
  processed_train_csv: "{PROCESSED_ROOT}/train.csv"
  processed_val_csv: "{PROCESSED_ROOT}/val.csv"
  processed_test_csv: "{PROCESSED_ROOT}/test.csv"
  output_dir: "/kaggle/temp/output"
  checkpoint_dir: "{CHECKPOINT_ROOT}"
  gcs_bucket: "{GCS_BUCKET}"
  gcs_project: "{GCS_PROJECT}"
wandb:
  entity: "phuongnm150505-uit"
  project: "meta-cxr-encoder-comparison"
java:
  home: "{java_home}"
  path: "{java_path}"
''')

print('PROJECT_DIR    =', PROJECT_DIR)
print('IMAGE_ROOT     =', IMAGE_ROOT)
print('PROCESSED_ROOT =', PROCESSED_ROOT)
print('CHECKPOINT_ROOT=', CHECKPOINT_ROOT)

PROJECT_DIR    = /kaggle/working/META-CXR
IMAGE_ROOT     = /kaggle/input/datasets/phuong20052/mimic-cxr-jpg-lite
PROCESSED_ROOT = /kaggle/input/datasets/phuong20052/mimic-cxr-p10-processed
CHECKPOINT_ROOT= /kaggle/temp/checkpoints


## Cell 2.5 — GCS auth + lazy checkpoint download

Đọc service-account JSON từ Kaggle Secrets (`GCS_SERVICE_ACCOUNT`, hoặc `GCP_SERVICE_ACCOUNT_JSON`/`GCP_SERVICE_ACCOUNT_B64`), tạo Google Cloud Storage client, rồi định nghĩa `ensure_local_checkpoint(run)` — chỉ tải `checkpoint_best.pth` khi chưa có local copy. Trả về `None` nếu trên GCS không tồn tại `checkpoint_best.pth`.

In [10]:
import base64
import json as _json
import os
from pathlib import Path

from google.cloud import storage
from google.oauth2 import service_account

GCS_PROJECT = os.environ.get("GCS_PROJECT", "mimic-cxr-jpg-491409")
GCS_BUCKET_NAME = os.environ.get("GCS_BUCKET", "meta-cxr-checkpoint").replace("gs://", "").rstrip("/")
GCS_BUCKET = f"gs://{GCS_BUCKET_NAME}"
CHECKPOINT_FILENAME = "checkpoint_best.pth"


def _get_secret(name):
    if os.environ.get(name):
        return os.environ[name]
    try:
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        return user_secrets.get_secret(name)

    except Exception:
        return None


def _load_service_account_info():
    raw = _get_secret("GCS_SERVICE_ACCOUNT") or _get_secret("GCP_SERVICE_ACCOUNT_JSON") or _get_secret("GCP_SERVICE_ACCOUNT_B64")
    if not raw:
        return None

    raw = raw.strip()
    try:
        return _json.loads(raw)
    except _json.JSONDecodeError:
        return _json.loads(base64.b64decode(raw).decode("utf-8"))


def build_storage_client(required=True):
    info = _load_service_account_info()
    if info:
        credentials = service_account.Credentials.from_service_account_info(info)
        return storage.Client(project=GCS_PROJECT, credentials=credentials)

    adc_path = os.environ.get("GOOGLE_APPLICATION_CREDENTIALS")
    if adc_path and os.path.exists(adc_path):
        return storage.Client(project=GCS_PROJECT)

    if required:
        raise RuntimeError(
            "GCS credentials not found. Add Kaggle Secret GCS_SERVICE_ACCOUNT "
            "with a service-account JSON key or base64-encoded JSON. "
            "Also accepted: GCP_SERVICE_ACCOUNT_JSON or GCP_SERVICE_ACCOUNT_B64. "
            f"The service account needs read access to {GCS_BUCKET}."
        )
    return None


def find_gcs_checkpoint_blob(client, run: str, filename: str = CHECKPOINT_FILENAME):
    candidates = []
    for blob in client.list_blobs(GCS_BUCKET_NAME, prefix=f"{run}/"):
        if blob.name == f"{run}/{filename}" or blob.name.endswith(f"/{filename}"):
            candidates.append(blob)
    if not candidates:
        return None
    return sorted(candidates, key=lambda b: ((b.updated.timestamp() if b.updated else 0), b.name))[-1]


storage_client = build_storage_client(required=True)
print(f"GCS bucket: {GCS_BUCKET}")
print(f"GCS project: {GCS_PROJECT}")


def ensure_local_checkpoint(run: str):
    """Ensure CHECKPOINT_ROOT/run/checkpoint_best.pth exists locally, downloading from GCS if needed."""
    local_dir = CHECKPOINT_ROOT / run
    local_dir.mkdir(parents=True, exist_ok=True)
    local = local_dir / CHECKPOINT_FILENAME
    if local.exists():
        return local

    blob = find_gcs_checkpoint_blob(storage_client, run, CHECKPOINT_FILENAME)
    if blob is None:
        return None

    blob.download_to_filename(str(local))
    print(f"Downloaded checkpoint: {GCS_BUCKET}/{blob.name} -> {local}")
    return local


GCS bucket: gs://meta-cxr-checkpoint
GCS project: mimic-cxr-jpg-491409


In [11]:
import gc
from types import SimpleNamespace

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import f1_score
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

import model.lavis.tasks as tasks
from model.lavis.common.config import Config
from model.lavis.common.registry import registry

# Registration imports.
from model.lavis.common.optims import LinearWarmupCosineLRScheduler, LinearWarmupStepLRScheduler
from model.lavis.datasets.builders import *
from model.lavis.models import *
from model.lavis.processors import *
from model.lavis.tasks import *
from model.lavis.data.ReportDataset import MIMIC_CXR_Dataset
from local_config import VIS_ROOT

registry.mapping['paths']['cache_root'] = '.'

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
EVAL_BATCH_SIZE = 4
NUM_WORKERS = 2

CHEXPERT_COLS = [
    'No Finding', 'Enlarged Cardiomediastinum', 'Cardiomegaly', 'Lung Opacity',
    'Lung Lesion', 'Edema', 'Consolidation', 'Pneumonia', 'Atelectasis',
    'Pneumothorax', 'Pleural Effusion', 'Pleural Other', 'Fracture', 'Support Devices',
]

FIVE_COMMON_ABNORMALITIES = [
    'Atelectasis',
    'Cardiomegaly',
    'Consolidation',
    'Edema',
    'Pleural Effusion',
]
TASK_IDXS = [CHEXPERT_COLS.index(name) for name in FIVE_COMMON_ABNORMALITIES]

print('DEVICE =', DEVICE)
print('5 tasks =', FIVE_COMMON_ABNORMALITIES)

/usr/local/lib/python3.12/dist-packages/timm/models/hub.py:4: FutureWarning: Importing from timm.models.hub is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)
/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


DEVICE = cuda
5 tasks = ['Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Pleural Effusion']


In [12]:
TABLE_RUNS = [
    {'run': '01_biovil_only',        'RN50': True,  'ViT': False, 'Swin': False},
    {'run': '02_pubmedclip_only',    'RN50': False, 'ViT': True,  'Swin': False},
    {'run': '03_swin_only',          'RN50': False, 'ViT': False, 'Swin': True},
    {'run': '04_biovil_pubmedclip',  'RN50': True,  'ViT': True,  'Swin': False},
    {'run': '05_biovil_swin',        'RN50': True,  'ViT': False, 'Swin': True},
    {'run': '06_pubmedclip_swin',    'RN50': False, 'ViT': True,  'Swin': True},
    {'run': '07_all_three',          'RN50': True,  'ViT': True,  'Swin': True},
]

# Giá trị paper Table 5 (paper liệt kê 6/7 combo; 06_pubmedclip_swin không có).
PAPER_F1 = {
    '01_biovil_only':       0.602,
    '02_pubmedclip_only':   0.473,
    '03_swin_only':         0.467,
    '04_biovil_pubmedclip': 0.631,
    '05_biovil_swin':       0.682,
    '06_pubmedclip_swin':   None,
    '07_all_three':         0.701,
}

def build_cfg(run_name: str):
    cfg_path = PROJECT_DIR / 'pretraining' / 'configs' / 'encoder_comparison' / f'{run_name}.yaml'
    args = SimpleNamespace(cfg_path=str(cfg_path), options=None)
    return Config(args)

def load_torch_checkpoint(path: Path):
    try:
        return torch.load(path, map_location='cpu', weights_only=False)
    except TypeError:
        return torch.load(path, map_location='cpu')


def build_model_for_run(run_name: str, checkpoint_path: Path):
    cfg = build_cfg(run_name)
    task = tasks.setup_task(cfg)
    model = task.build_model(cfg)
    ckpt = load_torch_checkpoint(checkpoint_path)
    state_dict = ckpt['model'] if isinstance(ckpt, dict) and 'model' in ckpt else ckpt
    missing, unexpected = model.load_state_dict(state_dict, strict=False)
    print(f'{run_name}: loaded {checkpoint_path.name}; missing={len(missing)}, unexpected={len(unexpected)}')
    model.to(DEVICE)
    model.eval()
    return cfg, model

def make_test_loader(cfg):
    dataset = MIMIC_CXR_Dataset(
        vis_processor=None,
        text_processor=None,
        vis_root=VIS_ROOT,
        split='test',
        cfg=cfg,
        truncate=None,
    )
    return DataLoader(
        dataset,
        batch_size=EVAL_BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE == 'cuda'),
    )

@torch.no_grad()
def predict_logits_with_text(model, batch):
    image = batch['image'].to(DEVICE, non_blocking=True)
    text = batch['text_output']

    cnn_patches, vit_patches, swin_patches, _ = model._encode_image_streams(image, apply_aug=False)
    text_tokens = model.tokenizer(
        text,
        padding='max_length',
        truncation=True,
        max_length=model.max_txt_len,
        return_tensors='pt',
    ).to(DEVICE)
    text_output = model.Qformer.bert(
        text_tokens.input_ids,
        attention_mask=text_tokens.attention_mask,
        return_dict=True,
    )
    logits, _, _, _, _ = model.mhcac(
        cnn_patches=cnn_patches,
        vit_patches=vit_patches,
        swin_patches=swin_patches,
        text_embeddings=text_output.last_hidden_state,
        labels=None,
    )
    return logits

def mean_f1_for_run(run_name: str):
    checkpoint_path = ensure_local_checkpoint(run_name)
    if checkpoint_path is None:
        raise FileNotFoundError(
            f'{run_name}: chưa có checkpoint_best.pth trên GCS '
            f'({GCS_BUCKET}/{run_name}/checkpoint_best.pth).'
        )
    cfg, model = build_model_for_run(run_name, checkpoint_path)
    loader = make_test_loader(cfg)

    all_preds = []
    all_labels = []
    for batch in tqdm(loader, desc=run_name):
        logits = predict_logits_with_text(model, batch)
        preds = torch.argmax(torch.softmax(logits, dim=-1), dim=-1)
        all_preds.append(preds[:, TASK_IDXS].cpu().numpy())
        all_labels.append(batch['classification_labels'][:, TASK_IDXS].cpu().numpy())

    y_pred = np.concatenate(all_preds, axis=0)
    y_true = np.concatenate(all_labels, axis=0)

    per_task = {}
    for col_idx, task_name in enumerate(FIVE_COMMON_ABNORMALITIES):
        per_task[task_name] = f1_score(
            y_true[:, col_idx],
            y_pred[:, col_idx],
            average='weighted',
            zero_division=1,
        )
    mean_f1 = float(np.mean(list(per_task.values())))

    del model, loader
    gc.collect()
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()

    return mean_f1, per_task, checkpoint_path

In [ ]:
rows = []
details = {}
skipped = []

for item in TABLE_RUNS:
    run_name = item['run']
    paper_value = PAPER_F1[run_name]
    try:
        mean_f1, per_task, checkpoint_path = mean_f1_for_run(run_name)
        details[run_name] = {
            'mean_f1': mean_f1,
            'per_task': per_task,
            'checkpoint': str(checkpoint_path),
        }
        rows.append({
            'RN50': '✓' if item['RN50'] else '–',
            'ViT': '✓' if item['ViT'] else '–',
            'Swin': '✓' if item['Swin'] else '–',
            'Mean F1 Score': round(mean_f1, 4),
            'Paper F1': paper_value if paper_value is not None else '—',
            'Delta vs Paper': round(mean_f1 - paper_value, 4) if paper_value is not None else '—',
        })
    except FileNotFoundError as exc:
        print(f'WARNING: skipping {run_name}: {exc}')
        skipped.append(run_name)
        rows.append({
            'RN50': '✓' if item['RN50'] else '–',
            'ViT': '✓' if item['ViT'] else '–',
            'Swin': '✓' if item['Swin'] else '–',
            'Mean F1 Score': 'MISSING',
            'Paper F1': paper_value if paper_value is not None else '—',
            'Delta vs Paper': '—',
        })

table = pd.DataFrame(rows, columns=['RN50', 'ViT', 'Swin', 'Mean F1 Score', 'Paper F1', 'Delta vs Paper'])
table.to_csv('/kaggle/working/encoder_mean_f1_table.csv', index=False)

print()
print('Skipped runs (no checkpoint_best.pth on GCS):', skipped or 'none')
print()

display(
    table.style
    .hide(axis='index')
    .set_caption('Mean F1 Score Across 5 Common Abnormalities on MIMIC-CXR Test Set (GCS checkpoints)')
)

table

Downloaded checkpoint: gs://meta-cxr-checkpoint/01_biovil_only/checkpoint_best.pth -> /kaggle/temp/checkpoints/01_biovil_only/checkpoint_best.pth


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

...

Downloaded checkpoint: gs://meta-cxr-checkpoint/06_pubmedclip_swin/checkpoint_best.pth -> /kaggle/temp/checkpoints/06_pubmedclip_swin/checkpoint_best.pth


/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Some weights of BertLMHeadModel were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['bert.encoder.layer.0.crossattention.output.LayerNorm.bias', 'bert.encoder.layer.0.crossattention.output.LayerNorm.weight', 'bert.encoder.layer.0.crossattention.output.dense.bias', 'bert.encoder.layer.0.crossattention.output.dense.weight', 'bert.encoder.layer.0.crossattention.self.key.bias', 'bert.encoder.layer.0.crossattention.self.key.weight', 'bert.encoder.layer.0.crossattention.self.query.bias', 'bert.encoder.layer.0.crossattention.self.query.weight', 'bert.encoder.layer.0.

06_pubmedclip_swin: loaded checkpoint_best.pth; missing=569, unexpected=0
Number of chexpert records: 227827
Number of annotation records: 2042
Number of annotation records: 2042
setting up scorers...


/kaggle/working/META-CXR/model/lavis/data/ReportDataset.py:261: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  self.chexpert['No Finding'].fillna(0.0, inplace=True)
/kaggle/working/META-CXR/model/lavis/data/ReportDataset.py:258: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For exam

06_pubmedclip_swin:   0%|          | 0/511 [00:00<?, ?it/s]